# Natural Language Processing - SemEval 2025 Task 9
### Team - Spaghetti & Spaetzle AI 
Tommaso Bergonzoni (Exchange Student, Italy) 

Simon Muehlbauer (Exchange Student, Germany)



### Libraries and Imports

In [2]:
import os
print(os.getcwd())
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import argparse
import pandas as pd
from datasets import load_dataset, Dataset
import os
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from torch.utils.data import DataLoader
from torch.optim import AdamW
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")




c:\Univeristy\Overseas\Natural-Language-Processing\Project


C:\Users\tomma\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Subtask 1


Constants

In [ ]:
ITA_DATA_1 = "data/subtask1/train/ita.csv"
DEU_DATA_1 = "data/subtask1/train/deu.csv"
ITA_DATA_2 = "data/subtask2/train/ita.csv"
DEU_DATA_2 = "data/subtask2/train/deu.csv"
MODEL_NAME = "xlm-roberta-base"
LR = 2e-5
BATCH_SIZE = 32
EPOCHS = 4
WD = 0.01

### Dataset

IMPORTANT: The dataset provided by the organization has a mistake. The "dev" set has 2 labels declared: "id, text, label" but has only 2 fields in all the dataset. 

As a solution, we derived train, val e test set directly from what the organization call "train set". 


In [15]:
def load_data(path_ita=ITA_DATA_1, path_deu=DEU_DATA_1):
    print("Reading it_data....")
    df_ita_train = pd.read_csv(path_ita)
    print("Reading deu_data....")
    df_deu_train = pd.read_csv(path_deu)
    return df_ita_train, df_deu_train

df_ita, df_deu = load_data()

print("\n\nItalian Data Head:")
df_ita.head()




Reading it_data....
Reading deu_data....


Italian Data Head:


,id,text,polarization
0,ita_406960551ce82c2d59ec051a4e5c12eb,#Buongiorno a tutti gli Italiani e agli strani...,0
1,ita_4607767f5654ead99d237973cb1009cc,"@URL questo è ciò che corano le tue ""RISORSE ""...",1
2,ita_db8d7735f59ed675a35bf2022f546901,"500ml in 6 anni, l'europa vuole che ci occupia...",0
3,ita_f6b6c6a2e983267538212426e1793386,"Il terrorista di Berlino ucciso a Milano, era ...",0
4,ita_2ac8d11d38b781ed8790a2c99a439e34,Sentire dire che contro il terrorismo dobbiamo...,1


Operate the following modifications: 
- Extract train, val e test sets from the main dataset
- For all the datasets, delete the column "id" which is not useful for our purpose
- Merge ita_train and deu_train in a single training dataset
- Merge ita_val and deu_val in a single training dataset
- Rename "polarization" to "label"

In [5]:
#Dropping column id
df_ita.drop(columns=['id'], inplace=True)
df_deu.drop(columns=['id'], inplace=True)

#Splitting datasets 
df_ita_train, df_ita_temp = train_test_split(
    df_ita,
    test_size=0.2,          # 20% validation+test
    stratify=df_ita['polarization']
)

df_ita_val, df_ita_test = train_test_split(
    df_ita_temp,
    test_size=0.5,          # 50% each for validation and test
    random_state=42,
    stratify=df_ita_temp['polarization']
)

df_deu_train, df_deu_temp = train_test_split(
    df_deu,
    test_size=0.2,          # 20% validation+test
    stratify=df_deu['polarization']
)

df_deu_val, df_deu_test = train_test_split(
    df_deu_temp,
    test_size=0.5,          # 50% each for validation and test
    random_state=42,
    stratify=df_deu_temp['polarization']
)

#Concat ita and deu for train and val sets
df_train = pd.concat([df_ita_train, df_deu_train], ignore_index=True)
df_val = pd.concat([df_ita_val, df_deu_val], ignore_index=True)

#renaming polarization to label
df_train = df_train.rename(columns={"polarization": "label"})
df_val = df_val.rename(columns={"polarization": "label"})
df_ita_test = df_ita_test.rename(columns={"polarization": "label"})
df_deu_test = df_deu_test.rename(columns={"polarization": "label"})

#removing NaN values
df_train = df_train.dropna(subset=["text", "label"])
df_val = df_val.dropna(subset=["text", "label"])
df_ita_test = df_ita_test.dropna(subset=["text", "label"])
df_deu_test = df_deu_test.dropna(subset=["text", "label"])

#Converting text in string
df_train["text"] = df_train["text"].astype(str)
df_val["text"] = df_val["text"].astype(str)
df_ita_test["text"] = df_ita_test["text"].astype(str)
df_deu_test["text"] = df_deu_test["text"].astype(str)

#Printing dataset sizes
print(f"Train set size: {len(df_train)}")
print(f"Validation set size: {len(df_val)}")
print(f"Italian Test set size: {len(df_ita_test)}")
print(f"German Test set size: {len(df_deu_test)}")

Train set size: 5211
Validation set size: 651
Italian Test set size: 334
German Test set size: 318


### Baseline Model

In [6]:
print("Dowloading Tokenizer and Model " + MODEL_NAME)

#Tokenizer from pretrained model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#Model from pretrained model with classification head
classification_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,        #Task 1 requires only 1 label (binary classification)
)

print("\n\nModel Informations:\n")
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("\n")
print(classification_model.config)

Dowloading Tokenizer and Model xlm-roberta-base


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.




Model Informations:

Tokenizer vocab size: 250002


XLMRobertaConfig {
  "architectures": [
    "XLMRobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 250002
}



### Tokenized dataset

In [7]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256, 
    )


# Convert from pandas and clean index column
def safe_from_pandas(df):
    ds = Dataset.from_pandas(df)
    # Rimuove la colonna solo se esiste
    if "__index_level_0__" in ds.column_names:
        ds = ds.remove_columns(["__index_level_0__"])
    return ds

ds_train = safe_from_pandas(df_train)
ds_val = safe_from_pandas(df_val)
ds_test_it = safe_from_pandas(df_ita_test)
ds_test_de = safe_from_pandas(df_deu_test)

#Tokenizing Datasets
ds_train_tok = ds_train.map(tokenize_batch, batched=True)
ds_val_tok = ds_val.map(tokenize_batch, batched=True)
ds_test_it_tok = ds_test_it.map(tokenize_batch, batched=True)
ds_test_de_tok = ds_test_de.map(tokenize_batch, batched=True)

# Format for PyTorch
cols = ["input_ids", "attention_mask", "label"]
ds_train_tok.set_format(type="torch", columns=cols)
ds_val_tok.set_format(type="torch", columns=cols)
ds_test_it_tok.set_format(type="torch", columns=cols)
ds_test_de_tok.set_format(type="torch", columns=cols)

#PRint first line
print("First line of ds_train_tok in pytorch format:")
print(ds_train_tok[0])

Map: 100%|██████████| 318/318 [00:00<00:00, 6308.81 examples/s]

First line of ds_train_tok in pytorch format:
{'label': tensor(0), 'input_ids': tensor([     0,    468, 116742,    408,  37526,     23,    468,   7763,      7,
         31519,   1380,  54744,  32656,     45, 194704,    118,    566,  43215,
          5400, 230742,     45,  69317,     23,    243,     31,      4,  21712,
            27,   1374,  95538,      2,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1, 

## Fine-tuning

In [8]:
# Dataloaders
train_loader = DataLoader(ds_train_tok, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(ds_val_tok,   batch_size=BATCH_SIZE, shuffle=True)

#OPtimizer and Loss function
optimizer = AdamW(classification_model.parameters(), lr=LR, weight_decay=WD)
criterion = torch.nn.CrossEntropyLoss()

In [9]:
#Training loop
print("Starting training...")
for epoch in range(EPOCHS):
    print("Training Epoch: ", epoch+1)
    classification_model.train()
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        mask      = batch["attention_mask"].to(device)
        labels    = batch["label"].to(device)
        optimizer.zero_grad()
        outputs = classification_model(input_ids, attention_mask=mask)[0]
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    # validazione
    classification_model.eval()
    all_preds, all_gold = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            mask      = batch["attention_mask"].to(device)
            labels    = batch["label"].to(device)
            outputs   = classification_model(input_ids, attention_mask=mask)[0]
            preds     = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_gold.extend(labels.cpu().numpy())
    acc = accuracy_score(all_gold, all_preds)
    f1  = f1_score(all_gold, all_preds)
    print(f"Epoch {epoch+1}/{EPOCHS} | Acc={acc:.4f} | F1={f1:.4f}")

Starting training...
Training Epoch:  1


KeyboardInterrupt: 

### Test

In [10]:
def evaluate_model(model, loader, lang_name=""):
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            outputs = model(input_ids, attention_mask=mask)[0]
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds)
    print(f"\n=== Test Results ({lang_name}) ===")
    print(f"Accuracy: {acc:.4f} | F1: {f1:.4f}")
    print(classification_report(all_labels, all_preds, digits=4))


classification_model.eval()
all_preds, all_labels = [], []

test_loader_it = DataLoader(ds_test_it_tok, batch_size=BATCH_SIZE)
test_loader_de = DataLoader(ds_test_de_tok, batch_size=BATCH_SIZE)


# Test ITA
print("Operating Italian Test......")
evaluate_model(classification_model, test_loader_it, "Italian")

# Test DEU
print("")
print("")
print("Operating German Test.....")
evaluate_model(classification_model, test_loader_de, "German")

Operating Italian Test......

=== Test Results (Italian) ===
Accuracy: 0.4102 | F1: 0.5817
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000       197
           1     0.4102    1.0000    0.5817       137

    accuracy                         0.4102       334
   macro avg     0.2051    0.5000    0.2909       334
weighted avg     0.1682    0.4102    0.2386       334



Operating German Test.....


C:\Users\tomma\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\tomma\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\tomma\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_clas

KeyboardInterrupt: 

## Improved Model

TODO.....

# Subtask 2


### Dataset

In [19]:
df_ita, df_deu = load_data(path_ita=ITA_DATA_2, path_deu=DEU_DATA_2)

print("\n\nItalian Data Head:")
df_ita.head()

Reading it_data....
Reading deu_data....


Italian Data Head:


,id,text,political,racial/ethnic,religious,gender/sexual,other
0,ita_406960551ce82c2d59ec051a4e5c12eb,#Buongiorno a tutti gli Italiani e agli strani...,0,0,0,0,0
1,ita_4607767f5654ead99d237973cb1009cc,"@URL questo è ciò che corano le tue ""RISORSE ""...",0,0,1,0,0
2,ita_db8d7735f59ed675a35bf2022f546901,"500ml in 6 anni, l'europa vuole che ci occupia...",0,0,0,0,0
3,ita_f6b6c6a2e983267538212426e1793386,"Il terrorista di Berlino ucciso a Milano, era ...",0,0,0,0,0
4,ita_2ac8d11d38b781ed8790a2c99a439e34,Sentire dire che contro il terrorismo dobbiamo...,0,0,1,0,0


Threathing the datasets similarly as before:

In [ ]:
#Dropping column id
df_ita.drop(columns=['id'], inplace=True)
df_deu.drop(columns=['id'], inplace=True)

#Splitting datasets 
df_ita_train, df_ita_temp = train_test_split(
    df_ita,
    test_size=0.2,          # 20% validation+test
)

df_ita_val, df_ita_test = train_test_split(
    df_ita_temp,
    test_size=0.5,          # 50% each for validation and test
    random_state=42,
)

df_deu_train, df_deu_temp = train_test_split(
    df_deu,
    test_size=0.2,          # 20% validation+test
)

df_deu_val, df_deu_test = train_test_split(
    df_deu_temp,
    test_size=0.5,          # 50% each for validation and test
    random_state=42,
)

#Concat ita and deu for train and val sets
df_train = pd.concat([df_ita_train, df_deu_train], ignore_index=True)
df_val = pd.concat([df_ita_val, df_deu_val], ignore_index=True)

#removing NaN values
df_train = df_train.dropna(subset=["text", "political", "racial/ethnic", "religious", "gender/sexual", "other"])
df_val = df_val.dropna(subset=["text", "political", "racial/ethnic", "religious", "gender/sexual", "other"])
df_ita_test = df_ita_test.dropna(subset=["text", "political", "racial/ethnic", "religious", "gender/sexual", "other"])
df_deu_test = df_deu_test.dropna(subset=["text", "political", "racial/ethnic", "religious", "gender/sexual", "other"])

#Converting text in string
df_train["text"] = df_train["text"].astype(str)
df_val["text"] = df_val["text"].astype(str)
df_ita_test["text"] = df_ita_test["text"].astype(str)
df_deu_test["text"] = df_deu_test["text"].astype(str)

#Printing dataset sizes
print(f"Train set size: {len(df_train)}")
print(f"Validation set size: {len(df_val)}")
print(f"Italian Test set size: {len(df_ita_test)}")
print(f"German Test set size: {len(df_deu_test)}")

Train set size: 5211
Validation set size: 651
Italian Test set size: 334
German Test set size: 318


## Baseline Model


Using the following script we can understand that the problem is multi-label. There is at least one instance which belogns to more than one class. For that reason we'll set problem_type = "multi_label_classification" during the model load. 

In [ ]:
multi_label = (df_ita[["political", "racial/ethnic", "religious", "gender/sexual", "other"]].sum(axis=1) > 1).any()
print("Is the problem multi-label ??", multi_label)


is the problem multi-label ?? True


In [24]:
print("Dowloading Tokenizer and Model " + MODEL_NAME)

#Tokenizer from pretrained model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#Model from pretrained model with classification head
classification_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5,
    problem_type="multi_label_classification",  
)

print("\n\nModel Informations:\n")
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("\n")
print(classification_model.config)

Dowloading Tokenizer and Model xlm-roberta-base


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.




Model Informations:

Tokenizer vocab size: 250002


XLMRobertaConfig {
  "architectures": [
    "XLMRobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 4
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "problem_type": "multi_label_classification",
  "transformers_version": "4.57.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size"

## Tokenize

In [25]:
# Create 'label' column as list of 5 elements and drop original columns
label_cols = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]
for df in [df_train, df_val, df_ita_test, df_deu_test]:
    df["label"] = df[label_cols].values.tolist()
    df.drop(columns=label_cols, inplace=True)

#Delete unused variables
ds_train = safe_from_pandas(df_train)
ds_val = safe_from_pandas(df_val)
ds_test_it = safe_from_pandas(df_ita_test)
ds_test_de = safe_from_pandas(df_deu_test)

#Tokenizing Datasets
ds_train_tok = ds_train.map(tokenize_batch, batched=True)
ds_val_tok = ds_val.map(tokenize_batch, batched=True)
ds_test_it_tok = ds_test_it.map(tokenize_batch, batched=True)
ds_test_de_tok = ds_test_de.map(tokenize_batch, batched=True)

# Format for PyTorch
cols = ["input_ids", "attention_mask", "label"]
ds_train_tok.set_format(type="torch", columns=cols)
ds_val_tok.set_format(type="torch", columns=cols)
ds_test_it_tok.set_format(type="torch", columns=cols)
ds_test_de_tok.set_format(type="torch", columns=cols)

import datasets
for ds in [ds_train_tok, ds_val_tok, ds_test_it_tok, ds_test_de_tok]:
    ds = ds.cast_column("label", datasets.Sequence(datasets.Value("float32")))

print("First line of ds_train_tok in pytorch format:")
print(ds_train_tok[0])


Casting the dataset: 100%|██████████| 318/318 [00:00<00:00, 57687.33 examples/s]

First line of ds_train_tok in pytorch format:
{'label': tensor([0, 0, 0, 1, 0]), 'input_ids': tensor([     0,   2371, 149039,    516,   2639,    117,    351,  73572,   1506,
           118,  41573,      5,     52,   4613,    565,  29639,   1854,  23992,
          5802,   1526,  29639,     45,    120,  10724,  22995,  41724,    351,
           739,   5213,  18435,  35931,     17,  68052, 182027,     23,     51,
         38055,   7543,    324,   1337,  31917,    188,      4,  20788,    188,
            23,      6,  31292,    146,    377,  18339,      4,  10724,  22995,
         41724,    739,  11099,    188,    211, 160709,    158,    120,      5,
          2552,  13400,   8773,  29195,    290,     40,   9222,      5,    378,
         95538,    268,      2,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,   

## Fine-Tuning

In [ ]:
#Creating Dataloaders
train_loader = DataLoader(ds_train_tok, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(ds_val_tok,   batch_size=BATCH_SIZE, shuffle=False)

#Optimizer
optimizer = AdamW(classification_model.parameters(), lr=LR, weight_decay=WD)

#Loss Function
criterion = torch.nn.BCEWithLogitsLoss()


In [27]:
print("Starting training...")

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    classification_model.train()
    running_loss = 0.0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        mask      = batch["attention_mask"].to(device)
        labels    = batch["label"].to(device).float()  

        optimizer.zero_grad()
        logits = classification_model(input_ids, attention_mask=mask)[0]
        loss   = criterion(logits, labels)         
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Training Loss: {running_loss / len(train_loader):.4f}")

    classification_model.eval()
    all_preds, all_gold = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            mask      = batch["attention_mask"].to(device)
            labels    = batch["label"].to(device).float()

            logits = classification_model(input_ids, attention_mask=mask)[0]
            preds  = torch.sigmoid(logits)               
            preds  = (preds > 0.5).int()
            all_preds.extend(preds.cpu().numpy())
            all_gold.extend(labels.cpu().numpy())

    acc = accuracy_score(all_gold, all_preds)
    f1  = f1_score(all_gold, all_preds, average="macro") 
    print(f"→ Validation Acc={acc:.4f} | F1={f1:.4f}")


Starting training...

Epoch 1/4


KeyboardInterrupt: 

## Test

In [ ]:
def evaluate_model(model, loader, lang_name=""):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            mask      = batch["attention_mask"].to(device)
            labels    = batch["label"].to(device).float()  
            logits = model(input_ids, attention_mask=mask)[0]
            preds  = torch.sigmoid(logits)
            preds  = (preds > 0.5).int()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    #Metrics
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="macro")
    print(f"\n=== Test Results ({lang_name}) ===")
    print(f"Accuracy: {acc:.4f} | F1: {f1:.4f}")
    print(classification_report(
        all_labels, all_preds,
        target_names=["political", "racial/ethnic", "religious", "gender/sexual", "other"],
        digits=4
    ))


classification_model.eval()

test_loader_it = DataLoader(ds_test_it_tok, batch_size=BATCH_SIZE)
test_loader_de = DataLoader(ds_test_de_tok, batch_size=BATCH_SIZE)

print("Operating Italian Test......")
evaluate_model(classification_model, test_loader_it, "Italian")

print("\nOperating German Test......")
evaluate_model(classification_model, test_loader_de, "German")

Operating Italian Test......


KeyboardInterrupt: 

## Improved Model


TODO